# BrainDecoding — Google Colab GPU 秒级解包、混合精度训练与验证集可视化评估工作流

本 Notebook 运行在 **Google Colab GPU 实例**（如 L4 / A100 / T4）上：
1. 自动创建并检查 Google Drive 标准目录（`/content/drive/MyDrive/BrainDecoding/{archives,runs,data,models}`）；
2. 从 Google Drive 秒级解包单文件缓存归档 `archives/features_<modality>.tar` 到本地高速盘 `/content/local_cache`；
3. 在本地 `/content/studies_links` 建立与 CPU 预热阶段完全一致的软链接桥（确保缓存签名 100% 命中）；
4. 通过 `--config` 直接读取 `configs/training/*.yaml` 配置文件（支持 **阶段 1: 词级脑编码器**、**阶段 2.1: 词级脑编码器 + 16 层全局上下文 Transformer**、**阶段 2.2: 词级脑编码器 + 16 层局部窗口滚动修订**）；
5. 支持通过 `WINDOW_SECONDS`（或修改 YAML 中的 `workflow.window_seconds`）一键切换单字脑信号时间窗长度（如 `0.5s`、`0.75s`、`1.0s`、`1.25s`、`1.5s`、`2.0s`，直接从 50Hz 连续信号缓存动态切窗，无需重新打包缓存）；
6. 支持通过 `EXP_TAG`（输出目录标签）与 `WORD_EXP_TAG`（阶段 2 热启动所引用的阶段 1 实验标签）灵活组合不同超参数实验；
7. 训练完成后一键执行**验证集 (`val`) 全面评估**，并在 Notebook 内直观绘制多词表规模检索、分受试者表现、训练收敛曲线与跨阶段对比图表。

In [ ]:
# 1. 检查 GPU 并挂载 Google Drive、自动创建目录与配置环境变量
!nvidia-smi

import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('非 Colab 环境，跳过 drive.mount')

# ==================== 用户可修改参数 ====================
GIT_REPO_URL = 'https://github.com/wdongdong0708-jpg/BrainDecoding2.git'
GIT_BRANCH = 'master'
DRIVE_BASE_DIR = '/content/drive/MyDrive/BrainDecoding'

# 直接选择 configs/training/ 下的清晰 YAML 配置文件（超参数已针对 L4 GPU 调优）：
#   - 'configs/training/pallier2025_word_encoder.yaml'                              (法语 MEG 阶段 1:   词级脑编码器 CNN)
#   - 'configs/training/pallier2025_context_transformer.yaml'                       (法语 MEG 阶段 2.1: 词级脑编码器 + 16层全局上下文 Transformer)
#   - 'configs/training/pallier2025_rolling_revision.yaml'                          (法语 MEG 阶段 2.2: 词级脑编码器 + 16层局部窗口滚动修订)
#   - 'configs/training/siamese_debiased/pallier2025_siamese_debiased_revision.yaml' (法语 MEG 阶段 2.2: 孪生共模抵消 + 错位零化正则 + 预提取极速训练)
#   - 'configs/training/siamese_debiased/pallier2025_siamese_debiased_context.yaml'  (法语 MEG 阶段 2.1: 孪生共模抵消 + 错位零化正则 + 预提取极速训练)
#   - 'configs/training/chineseeeg2_word_encoder.yaml'                              (中文 EEG 阶段 1:   词级脑编码器 CNN)
#   - 'configs/training/chineseeeg2_context_transformer.yaml'                       (中文 EEG 阶段 2.1: 词级脑编码器 + 16层全局上下文 Transformer)
#   - 'configs/training/chineseeeg2_rolling_revision.yaml'                          (中文 EEG 阶段 2.2: 词级脑编码器 + 16层局部窗口滚动修订)
#   - 'configs/training/siamese_debiased/chineseeeg2_siamese_debiased_revision.yaml' (中文 EEG 阶段 2.2: 孪生共模抵消 + 错位零化正则 + 预提取极速训练)
CONFIG_FILE = 'configs/training/siamese_debiased/pallier2025_siamese_debiased_revision.yaml'
MODALITY = 'meg'                      # 对应归档标识: 'meg' (pallier2025) 或 'eeg' (chineseeeg2)
EXP_TAG = 'lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window'              # 当前训练输出实验标签（输出为 runs/<dataset>/<EXP_TAG>/<stage>/...；留空 '' 则为 runs/<dataset>/<stage>/...）
WORD_EXP_TAG = 'lr4e-4_bs256'                     # 阶段 2 (2.1/2.2) 热启动加载哪个实验标签下的阶段 1 (word) 权重（留空 '' 默认与 EXP_TAG 相同；填 'default' 加载无标签根目录 word）
WINDOW_SECONDS = 1                 # 单字脑信号时间窗长度（秒，如 0.5, 0.75, 1.0, 1.25, 1.5, 2.0；填 None 则沿用 YAML 中的 window_seconds，默认 1.0s）
WINDOW_START_OFFSET_SECONDS = None    # 相对于词起始时刻的窗口起点偏移（秒，默认 0.0；填 None 则沿用 YAML 配置）
ETA_TRAIN = 0.8                      # 可选覆写孪生共模扣除权重 eta_train（如 0.8 或 0.5；填 None 则沿用 YAML 配置；Stage 1 word 阶段自动忽略）
LAMBDA_NULL = None                    # 可选覆写错位零化正则系数 lambda_null（如 10.0；填 None 则沿用 YAML 配置；Stage 1 word 阶段自动忽略）
SHIFTS_PER_STEP = 6                # 可选覆写每步抽取错位分支数 M（如 2 或 6；填 None 则沿用 YAML 配置；Stage 1 word 阶段自动忽略）
SEED = '0,1,2'                        # 随机种子：支持单种子 (如 0、1、2) 或多种子一键连跑 (如 '0,1,2' 或 [0, 1, 2])
SUBSET_ARGS = '--subject all'         # 或 '--num-subjects 2' / '--subjects sub-01,sub-02'
# ========================================================

if isinstance(SEED, (list, tuple, set)):
    SEED_LIST = [int(s) for s in SEED]
else:
    SEED_LIST = [int(s.strip()) for s in str(SEED).strip('[]()').replace(' ', ',').split(',') if s.strip()]
SEED_ARG = ','.join(str(s) for s in SEED_LIST)

WINDOW_ARGS = f'--window-seconds {WINDOW_SECONDS}' if WINDOW_SECONDS is not None and str(WINDOW_SECONDS).strip() != '' else ''
if WINDOW_START_OFFSET_SECONDS is not None and str(WINDOW_START_OFFSET_SECONDS).strip() != '':
    WINDOW_ARGS = f'{WINDOW_ARGS} --window-start-offset-seconds {WINDOW_START_OFFSET_SECONDS}'.strip()

SIAMESE_ARGS = ''
_is_word_stage_cfg = 'word_encoder' in str(CONFIG_FILE).lower() or '/main_word' in str(CONFIG_FILE).lower()
if not _is_word_stage_cfg:
    if ETA_TRAIN is not None and str(ETA_TRAIN).strip() != '':
        SIAMESE_ARGS = f'{SIAMESE_ARGS} --eta-train {ETA_TRAIN}'.strip()
    if LAMBDA_NULL is not None and str(LAMBDA_NULL).strip() != '':
        SIAMESE_ARGS = f'{SIAMESE_ARGS} --lambda-null {LAMBDA_NULL}'.strip()
    if SHIFTS_PER_STEP is not None and str(SHIFTS_PER_STEP).strip() != '':
        SIAMESE_ARGS = f'{SIAMESE_ARGS} --shifts-per-step {SHIFTS_PER_STEP}'.strip()

CODE_DIR = '/content/code'
VENV_DIR = '/content/proj-venv'
PYTHON = f'{VENV_DIR}/bin/python'
LINKS_DIR = '/content/studies_links'
LOCAL_CACHE_DIR = '/content/local_cache'

DRIVE_DATA_DIR = f'{DRIVE_BASE_DIR}/data'
DRIVE_ARCHIVE_DIR = f'{DRIVE_BASE_DIR}/archives'
DRIVE_RESULTS_DIR = f'{DRIVE_BASE_DIR}/runs'
DRIVE_MODELS_DIR = f'{DRIVE_BASE_DIR}/models'

for folder in (
    LINKS_DIR,
    LOCAL_CACHE_DIR,
    DRIVE_DATA_DIR,
    DRIVE_ARCHIVE_DIR,
    DRIVE_RESULTS_DIR,
    DRIVE_MODELS_DIR,
):
    Path(folder).mkdir(parents=True, exist_ok=True)

# 统一通过环境变量驱动所有路径（与 CPU 预热完全对齐）
os.environ['BRAINDECODING_DATA_DIR'] = LINKS_DIR
os.environ['BRAINDATA_ROOT'] = LINKS_DIR
os.environ['BRAINDECODING_CACHE_DIR'] = LOCAL_CACHE_DIR
os.environ['BRAINDECODING_RESULTS_DIR'] = DRIVE_RESULTS_DIR
os.environ['BRAINDECODING_EXP_TAG'] = EXP_TAG
os.environ['BRAINDECODING_WORD_EXP_TAG'] = WORD_EXP_TAG
os.environ['BRAINDECODING_MODEL_DIR'] = DRIVE_MODELS_DIR

print('CONFIG_FILE               =', CONFIG_FILE)
print('WINDOW_SECONDS            =', WINDOW_SECONDS if WINDOW_SECONDS is not None else '(from YAML default 1.0s)')
print('BRAINDECODING_DATA_DIR    =', os.environ['BRAINDECODING_DATA_DIR'])
print('BRAINDECODING_CACHE_DIR   =', os.environ['BRAINDECODING_CACHE_DIR'])
print('BRAINDECODING_RESULTS_DIR =', os.environ['BRAINDECODING_RESULTS_DIR'])
print('BRAINDECODING_EXP_TAG     =', os.environ['BRAINDECODING_EXP_TAG'] or '(default)')
print('BRAINDECODING_WORD_EXP_TAG=', os.environ['BRAINDECODING_WORD_EXP_TAG'] or '(same as EXP_TAG / fallback default)')

In [ ]:
# 2. 克隆或同步最新代码到 /content/code
!if [ ! -d "{CODE_DIR}/.git" ]; then \
    git clone -b {GIT_BRANCH} {GIT_REPO_URL} {CODE_DIR}; \
else \
    git -C {CODE_DIR} fetch origin && git -C {CODE_DIR} checkout {GIT_BRANCH} && git -C {CODE_DIR} pull --ff-only; \
fi

In [ ]:
# 3. 防崩版初始化虚拟环境（锁定系统 PyTorch/CUDA 版本）+ 建立 /content/studies_links 软链接桥
!python3 -u {CODE_DIR}/scripts/colab_setup.py \
    --venv-dir {VENV_DIR} \
    --project-root {CODE_DIR} \
    --drive-data-dir {DRIVE_DATA_DIR} \
    --links-dir {LINKS_DIR} \
    --local-cache-dir {LOCAL_CACHE_DIR}

In [ ]:
# 4. 将 Google Drive 上的单一 .tar 缓存包秒级解包回 /content/local_cache
!cd {CODE_DIR} && {PYTHON} -u scripts/cache_archive.py unpack \
    --archive-dir {DRIVE_ARCHIVE_DIR} \
    --cache-dir {LOCAL_CACHE_DIR} \
    --modality {MODALITY} \
    --skip-existing

In [ ]:
# 5. 启动 GPU 训练（支持单种子或多种子连续训练 + 实时 tqdm 进度条 + 自动从 last.ckpt 断点续训）
# 输出路径：{DRIVE_RESULTS_DIR}/<dataset>/<EXP_TAG>/<stage>/<subset_tag>/seed-000 (及 seed-001, seed-002...)
# 若为阶段 2.1/2.2 且设置了 WORD_EXP_TAG，会自动从 {DRIVE_RESULTS_DIR}/<dataset>/<WORD_EXP_TAG>/word/.../best.ckpt 加载对应种子的阶段 1 权重
!cd {CODE_DIR} && {PYTHON} -u -m braindecoding train \
    --config {CONFIG_FILE} \
    --seed {SEED_ARG} \
    --require-cache \
    --resume auto \
    {WINDOW_ARGS} \
    {SIAMESE_ARGS} \
    {SUBSET_ARGS}

In [ ]:
# 6. 验证集 (val) 全面评估与可视化图表展示（多词表规模检索 + 分受试者精度 + 收敛曲线 + 跨阶段对比）
!cd {CODE_DIR} && {PYTHON} -u -m braindecoding eval-run \
    --config {CONFIG_FILE} \
    --seed {SEED_ARG} \
    --split val \
    --require-cache \
    {WINDOW_ARGS} \
    {SUBSET_ARGS}

from IPython.display import Image, display

for seed_val in SEED_LIST:
    eval_pngs = sorted(
        Path(DRIVE_RESULTS_DIR).rglob(f'seed-{int(seed_val):03d}/figures/*_val*.png'),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    shown = set()
    for png_path in eval_pngs:
        if EXP_TAG and EXP_TAG not in png_path.parts:
            continue
        if png_path.name in shown:
            continue
        shown.add(png_path.name)
        print(f'\n[图表预览 | seed-{int(seed_val):03d}] {png_path}')
        display(Image(filename=str(png_path)))

In [ ]:
# 最后一个单元格
import os
import time
from google.colab import drive, runtime

# 1. 刷新系统层面的文件写缓冲
os.sync()

# 2. 强制 Google Drive 将挂载目录中的所有缓存刷新并同步至云端
drive.flush_and_unmount()

# 3. 预留 5-10 秒缓冲时间，确保网络请求收尾
print("数据已全部同步至 Google Drive，正在释放运行时以停止扣费...")
time.sleep(10)

# 4. 释放实例
runtime.unassign()